# Super-resolution preprocessing (CodeFormer) best
increasing faces quality on an image

## 1 - Setup

In [ ]:
!nvidia-smi

# Shim for basicsr's broken torchvision import (run BEFORE installing/importing basicsr)
import torchvision.transforms.functional as F
import sys, types
shim = types.ModuleType("torchvision.transforms.functional_tensor")
shim.rgb_to_grayscale = F.rgb_to_grayscale
sys.modules["torchvision.transforms.functional_tensor"] = shim

!pip install -q basicsr facexlib

import basicsr

In [ ]:
import os
if not os.path.exists('CodeFormer'):
    !git clone https://github.com/sczhou/CodeFormer.git

%cd CodeFormer
!pip install -q -r requirements.txt
!python basicsr/setup.py develop
%cd ..

print("CodeFormer setup done.")

In [ ]:
import os
os.makedirs('CodeFormer/weights/CodeFormer', exist_ok=True)
os.makedirs('CodeFormer/weights/facelib', exist_ok=True)
os.makedirs('CodeFormer/weights/realesrgan', exist_ok=True)

!wget -q -O CodeFormer/weights/CodeFormer/codeformer.pth https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth
!wget -q -O CodeFormer/weights/facelib/detection_Resnet50_Final.pth https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/detection_Resnet50_Final.pth
!wget -q -O CodeFormer/weights/facelib/parsing_parsenet.pth https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/parsing_parsenet.pth
!wget -q -O CodeFormer/weights/realesrgan/RealESRGAN_x2plus.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x2plus.pth

print("Weights downloaded.")

## 2 - upload test image

In [ ]:
!wget https://raw.githubusercontent.com/Morteza-Asadi-Shalmaiy/Superres-fr/refs/heads/main/assets/superres-test-image-04.jpg /content

In [ ]:
input_path = "/content/superres-test-image-04.jpg"

import shutil
os.makedirs('CodeFormer/inputs/my_test', exist_ok=True)
shutil.copy(input_path, f'CodeFormer/inputs/my_test/input.jpg')

## 3 - body

In [ ]:
import glob

# Find and patch realesrgan_utils.py
files_to_patch = glob.glob('/content/CodeFormer/**/realesrgan_utils.py', recursive=True)
print("Found files:", files_to_patch)

for fpath in files_to_patch:
    with open(fpath, 'r') as f:
        content = f.read()

    old = "torch.load(model_path, map_location=torch.device('cpu'))"
    new = "torch.load(model_path, map_location=torch.device('cpu'), weights_only=False)"

    if old in content:
        content = content.replace(old, new)
        with open(fpath, 'w') as f:
            f.write(content)
        print(f"Patched: {fpath}")
    else:
        print(f"Pattern not found in: {fpath} (may already be patched)")

# Download RealESRGAN_x2plus.pth from HuggingFace mirror

import os

!rm -f CodeFormer/weights/realesrgan/RealESRGAN_x2plus.pth

# This HF repo mirrors the Real-ESRGAN release files
!wget -q --show-progress -O CodeFormer/weights/realesrgan/RealESRGAN_x2plus.pth "https://huggingface.co/ai-forever/Real-ESRGAN/resolve/main/RealESRGAN_x2.pth"

size = os.path.getsize('CodeFormer/weights/realesrgan/RealESRGAN_x2plus.pth')
print("Size:", size, "bytes")

if size < 1000000:
    print("Trying another mirror...")
    !rm -f CodeFormer/weights/realesrgan/RealESRGAN_x2plus.pth
    !wget -q --show-progress -O CodeFormer/weights/realesrgan/RealESRGAN_x2plus.pth "https://huggingface.co/dtarnow/UPscaler/resolve/main/RealESRGAN_x2plus.pth"
    size = os.path.getsize('CodeFormer/weights/realesrgan/RealESRGAN_x2plus.pth')
    print("Size:", size, "bytes")

# Re-run inference

%cd CodeFormer
!python inference_codeformer.py -w 0.7 --input_path inputs/my_test --output_path ../codeformer_results --face_upsample
%cd ..

In [ ]:
# Run CodeFormer

%cd CodeFormer
!python inference_codeformer.py -w 0.7 --input_path inputs/my_test --output_path ../codeformer_results
%cd ..

import glob
codeformer_files = glob.glob('codeformer_results/final_results/*')
print("Output files:", codeformer_files)

## 4 - show result

In [ ]:
import cv2
import matplotlib.pyplot as plt

original = cv2.imread(input_path)
codeformer_output = cv2.imread(codeformer_files[0])

def resize_to_height(img, height=300):
    ratio = height / img.shape[0]
    return cv2.resize(img, (int(img.shape[1] * ratio), height))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(resize_to_height(original), cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Original {original.shape[1]}x{original.shape[0]}")
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(resize_to_height(codeformer_output), cv2.COLOR_BGR2RGB))
axes[1].set_title(f"CodeFormer {codeformer_output.shape[1]}x{codeformer_output.shape[0]}")
axes[1].axis('off')

plt.tight_layout()
plt.show()